# 09 Autoformer NeuralForecast

This notebook evaluates a fixed B forecast using NeuralForecast's Autoformer implementation.

This is a library Autoformer baseline, separate from `autoformer_lite`. It is still a reference deep-learning baseline, not the main analysis model. The model is univariate, uses no exogenous variables, trains on `y = log(number_parcels)`, and is evaluated on the original `number_parcels` scale.

Note: On Windows + Python 3.13, `ray` may not be installable. NeuralForecast imports `ray` for auto-tuning classes even when fixed hyperparameters are used. The helper module installs minimal in-memory stubs for those auto-tuning imports, and this notebook does not use ray or hyperparameter tuning.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_connected_parcel_data
from src.forecasting.autoformer_neuralforecast import (
    MODEL_NAME,
    SPEC_NAME,
    fit_autoformer_neuralforecast,
    forecast_autoformer_neuralforecast,
)
from src.forecasting.evaluation import evaluate_forecasts
from src.forecasting.splits import make_fixed_split_b

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
FORECAST_DIR = PROJECT_ROOT / "output" / "forecasts"
PREDICTIONS_DIR = FORECAST_DIR / "predictions"
METRICS_DIR = FORECAST_DIR / "metrics"
FIGURES_DIR = FORECAST_DIR / "figures"

for path in [PREDICTIONS_DIR, METRICS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_NAME)
print("Spec:", SPEC_NAME)

Project root: C:\Users\fugat\Desktop\python_project\Transport_amount_project
Model: autoformer_neuralforecast
Spec: neuralforecast_autoformer_minimal


## 1. Load Data And Build fixed B

The connected monthly parcel series is loaded from `data/processed/parcel_volume_connected.csv`. fixed B trains on 2002-04 to 2023-12 and forecasts 2024-01 to 2026-02.

In [2]:
df = load_connected_parcel_data(str(DATA_PATH))
if "y" not in df.columns:
    df["y"] = np.log(df["number_parcels"].astype(float))

split = make_fixed_split_b(df)
train = split["train"]
test = split["test"]

split_summary = pd.DataFrame(
    [
        {
            "split": split["split"],
            "train_start": train.index.min().date(),
            "train_end": train.index.max().date(),
            "train_rows": len(train),
            "test_start": test.index.min().date(),
            "test_end": test.index.max().date(),
            "test_rows": len(test),
        }
    ]
)
display(split_summary)

,split,train_start,train_end,train_rows,test_start,test_end,test_rows
0,fixed_b,2002-04-01,2023-12-01,261,2024-01-01,2026-02-01,26


## 2. Fit NeuralForecast Autoformer

The model is intentionally small for CPU execution. Hyperparameter search is not performed. The moving-average window is set to 13 because the Autoformer decomposition uses a centered moving average and an odd window avoids length mismatch errors.

In [3]:
model_bundle = fit_autoformer_neuralforecast(
    train,
    target_col="y",
    horizon=len(test),
    input_size=24,
    hidden_size=16,
    n_head=2,
    encoder_layers=1,
    decoder_layers=1,
    conv_hidden_size=16,
    moving_avg_window=13,
    max_steps=200,
    random_seed=42,
)

fit_info = {
    **model_bundle.config,
    **{f"version_{key}": value for key, value in model_bundle.versions.items()},
    "train_seconds": model_bundle.train_seconds,
}
fit_summary = pd.DataFrame([fit_info])
display(fit_summary.T.rename(columns={0: "value"}))

Seed set to 42


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


C:\Users\fugat\Desktop\python_project\Transport_amount_project\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


,value
model,autoformer_neuralforecast
spec_name,neuralforecast_autoformer_minimal
target_col,y
target_scale_training,log
horizon,26
input_size,24
hidden_size,16
n_head,2
encoder_layers,1
decoder_layers,1


## 3. Forecast And Evaluate

Forecasts are produced on the log scale by NeuralForecast and transformed back with `exp()` before evaluation. The output is converted to the common forecast format used by the other models.

In [4]:
forecast_df = forecast_autoformer_neuralforecast(model_bundle, test, split="fixed_b")
metrics_df = evaluate_forecasts(forecast_df, y_train=train["number_parcels"])

display(forecast_df.head())
display(metrics_df)

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


C:\Users\fugat\Desktop\python_project\Transport_amount_project\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


,date,cutoff,horizon,y_true,y_pred,model,split,forecast_type,spec_name,target_col,target_scale
0,2024-01-01,2023-12-01,1,348749.0,402918.413383,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_minimal,number_parcels,original
1,2024-02-01,2023-12-01,2,344158.0,384099.488727,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_minimal,number_parcels,original
2,2024-03-01,2023-12-01,3,392055.0,402737.854969,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_minimal,number_parcels,original
3,2024-04-01,2023-12-01,4,369921.0,397824.947405,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_minimal,number_parcels,original
4,2024-05-01,2023-12-01,5,369040.0,406042.140230,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_minimal,number_parcels,original


,model,split,forecast_type,spec_name,n,rmse,mae,mape,mase
0,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_minimal,26,32929.373359,24639.602782,6.262312,1.970093


## 4. Compare With Existing fixed B Benchmarks

This comparison is only for orientation. `autoformer_neuralforecast` is an unconditional deep-learning baseline. It should mainly be compared with other unconditional models such as `autoformer_lite`, `seasonal_naive`, and Prophet without regressors.

In [5]:
comparison_rows = []
source_files = [
    "autoformer_lite_metrics.csv",
    "naive_metrics.csv",
    "prophet_metrics.csv",
]

for filename in source_files:
    path = METRICS_DIR / filename
    if not path.exists():
        print(f"skip: {filename} not found")
        continue
    frame = pd.read_csv(path)
    frame = frame[frame["split"] == "fixed_b"].copy()
    if filename == "naive_metrics.csv":
        frame = frame[frame["model"].isin(["seasonal_naive"])]
    if filename == "prophet_metrics.csv":
        frame = frame[frame["model"] == "prophet"]
    frame["source_file"] = filename
    comparison_rows.append(frame)

comparison_rows.append(metrics_df.assign(source_file="autoformer_neuralforecast_metrics.csv"))
comparison = pd.concat(comparison_rows, ignore_index=True, sort=False)
comparison = comparison[["model", "split", "forecast_type", "spec_name", "rmse", "mae", "mape", "mase", "source_file"]]
comparison = comparison.sort_values("rmse").reset_index(drop=True)
display(comparison)

,model,split,forecast_type,spec_name,rmse,mae,mape,mase,source_file
0,seasonal_naive,fixed_b,unconditional,post2020_m5,13622.783591,11619.923077,2.906761,0.929087,naive_metrics.csv
1,prophet,fixed_b,unconditional,prophet_no_regressors,26730.003403,19845.066371,4.834269,1.586739,prophet_metrics.csv
2,autoformer_neuralforecast,fixed_b,unconditional,neuralforecast_autoformer_minimal,32929.373359,24639.602782,6.262312,1.970093,autoformer_neuralforecast_metrics.csv
3,autoformer_lite,fixed_b,unconditional,autoformer_lite_univariate,35587.389423,32410.636313,8.536476,2.591436,autoformer_lite_metrics.csv


## 5. Save Outputs

The outputs are written under `output/forecasts/`. Existing model outputs are not overwritten.

In [6]:
forecast_path = PREDICTIONS_DIR / "fixed_b_autoformer_neuralforecast.csv"
metrics_path = METRICS_DIR / "autoformer_neuralforecast_metrics.csv"
fit_summary_path = METRICS_DIR / "autoformer_neuralforecast_fit_summary.csv"
figure_path = FIGURES_DIR / "fixed_b_autoformer_neuralforecast_forecast.png"

forecast_df.to_csv(forecast_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
fit_summary.to_csv(fit_summary_path, index=False)

fig, ax = plt.subplots(figsize=(10.5, 5.5))
train_tail = train.tail(24)
ax.plot(train_tail.index, train_tail["number_parcels"], color="0.55", linewidth=1.2, label="train actual tail")
ax.plot(test.index, test["number_parcels"], color="black", linewidth=1.6, label="test actual")
ax.plot(forecast_df["date"], forecast_df["y_pred"], marker="o", linewidth=1.2, label=MODEL_NAME)
ax.axvline(train.index.max(), color="0.2", linestyle=":", linewidth=1.0)
ax.set_title("fixed_b: NeuralForecast Autoformer forecast")
ax.set_xlabel("Date")
ax.set_ylabel("number_parcels")
ax.grid(True, color="0.85", linewidth=0.8)
ax.legend()
fig.tight_layout()
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print("saved", forecast_path)
print("saved", metrics_path)
print("saved", fit_summary_path)
print("saved", figure_path)

saved C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\predictions\fixed_b_autoformer_neuralforecast.csv
saved C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\metrics\autoformer_neuralforecast_metrics.csv
saved C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\metrics\autoformer_neuralforecast_fit_summary.csv
saved C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\figures\fixed_b_autoformer_neuralforecast_forecast.png


## 6. Interpretation Notes

- This is a NeuralForecast library Autoformer baseline, not the earlier local `autoformer_lite` implementation.
- It is fixed B only, univariate, and unconditional. No event dummies or exogenous regressors are used.
- The dataset has only 287 monthly observations, so weak performance or overfitting is not surprising.
- This is not the main analysis model. It is a developmental reference comparison for deep-learning-style forecasting.